0. Melihat data hasil predictive terlebih dahulu

In [3]:
import pandas as pd
from pathlib import Path

# Konfigurasi Visualisasi
pd.set_option('display.max_column',None)

# --- TEKNIK ROBUST: Dynamic Path Resolution ---
# 1. Mengambil lokasi folder saat ini secara dinamis
# Gunakan .cwd() untuk pathlib atau Path(__file__).parent jika di dalam script .py
current_dir = Path.cwd() 

# 2. Navigasi ke Root Project (Olist_Ecommerce_Analytics_Portfolio)
# Berdasarkan struktur folder Anda, dari notebooks/02_logistics_delivery/Research & Dev...
# kita perlu naik 3 tingkat untuk mencapai Root.
project_root = current_dir.parent.parent.parent

# 3. Definisikan path ke file parquet di folder production
data_path = project_root / "data" / "production" / "05_logistics_final_delivery_predictions.parquet"

# --- VERIFIKASI & LOADING ---
print(f"🔍 Mencari file di: {data_path}")

if data_path.exists():
    df = pd.read_parquet(data_path)
    print(f"✅ Berhasil memuat data!")
    print(f"📊 Shape Data: {df.shape}")
    display(df.head())
else:
    print(f"❌ File TIDAK ditemukan.")
    print(f"TIPS: Pastikan file '05_logistics_final_delivery_predictions.parquet' sudah ada di folder production.")

🔍 Mencari file di: c:\Users\etc\OneDrive\Documents\Marketing Data Analyst Portofolio\Olist_Ecommerce_Analytics_Portfolio\data\production\05_logistics_final_delivery_predictions.parquet
✅ Berhasil memuat data!
📊 Shape Data: (22037, 7)


,product_weight_g,product_volume_cm3,freight_value,customer_state,late_risk_probability,predicted_delay_days,action_plan
0,8750.0,45084.0,25.06,SP,0.506888,4.968518,⚠️ WARNING: Priority Warehouse Handling Required
1,750.0,9900.0,11.83,SP,0.317983,3.429360,✅ NORMAL: Standard Processing
2,700.0,32760.0,10.96,SP,0.274243,2.997217,✅ NORMAL: Standard Processing
3,300.0,816.0,7.78,SP,0.340756,2.414025,✅ NORMAL: Standard Processing
4,600.0,4800.0,12.79,SP,0.317910,3.200210,✅ NORMAL: Standard Processing


1. Optimization Setup & Data Ingestion
Menyiapkan lingkungan kerja dan memuat hasil prediksi risiko keterlambatan dari tahap sebelumnya.

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

# --- LANGKAH 1: SETUP OPTIMASI & PEMUATAN DATA ---
def setup_optimization_environment():
    project_root = Path.cwd().parent.parent.parent
    input_path = project_root / "data" / "production" / "05_logistics_final_delivery_predictions.parquet"
    
    if not input_path.exists():
        raise FileNotFoundError(f"Data prediksi final tidak ditemukan di: {input_path}")
        
    df_pred = pd.read_parquet(input_path)
    return df_pred, project_root

df_prescriptive, root = setup_optimization_environment()
print(f"✅ Data Prediksi termuat: {len(df_prescriptive):,} baris siap dioptimalkan.")

✅ Data Prediksi termuat: 22,037 baris siap dioptimalkan.


2. Cost-Benefit Framework Definition
Mendefinisikan parameter finansial untuk menghitung dampak ekonomi dari setiap keputusan logistik.

In [6]:
# --- LANGKAH 2: KERANGKA KERJA BIAYA-MANFAAT ---
class LogisticsFinanceConfig:
    # Estimasi biaya berdasarkan standar industri e-commerce
    CHURN_COST_PER_LATE = 50.0       # Kerugian nilai pelanggan jika paket telat (LTV loss)
    PREMIUM_SHIPPING_ADD_COST = 15.0 # Biaya tambahan upgrade ke kurir ekspres
    LATE_PENALTY_FEE = 10.0          # Penalti operasional per paket telat
    
    # Threshold probabilitas untuk aksi preskriptif
    CRITICAL_THRESHOLD = 0.8         # Risiko > 80% memerlukan tindakan drastis

3. Prescriptive Simulation Engine
Menggunakan pemrosesan vektor NumPy untuk mensimulasikan penghematan biaya secara massal.

In [7]:
# --- LANGKAH 3: MESIN SIMULASI PRESKRIPTIF ---
def run_optimization_simulation(df):
    # Menggunakan NumPy untuk performa tinggi pada 110k+ data
    probs = df['late_risk_probability'].values
    current_late = (probs > 0.5).astype(int)
    
    # Skenario: Upgrade ke Premium Shipping untuk risiko > 0.8
    # Menghitung potensi penghematan vs biaya tambahan
    is_upgraded = (probs > LogisticsFinanceConfig.CRITICAL_THRESHOLD)
    
    # Biaya Tanpa Optimasi: (Jumlah Telat * (Churn + Penalty))
    cost_baseline = np.sum(current_late * (LogisticsFinanceConfig.CHURN_COST_PER_LATE + LogisticsFinanceConfig.LATE_PENALTY_FEE))
    
    # Biaya Dengan Optimasi: (Upgraded * Premium_Cost) + (Remaining_Late * (Churn + Penalty))
    # Diasumsikan Premium Shipping memiliki tingkat keberhasilan 95% (on-time)
    cost_optimized = (np.sum(is_upgraded * LogisticsFinanceConfig.PREMIUM_SHIPPING_ADD_COST) + 
                      np.sum((current_late & ~is_upgraded) * (LogisticsFinanceConfig.CHURN_COST_PER_LATE + LogisticsFinanceConfig.LATE_PENALTY_FEE)))
    
    roi_saving = cost_baseline - cost_optimized
    return roi_saving, is_upgraded

total_savings, upgrade_list = run_optimization_simulation(df_prescriptive)
print(f"💰 Potensi Penghematan Biaya Logistik: R$ {total_savings:,.2f}")

💰 Potensi Penghematan Biaya Logistik: R$ 1,035.00


4. Optimization Logic & Strategic Constraints
Menerjemahkan angka menjadi rekomendasi taktis yang dapat dilakukan oleh tim operasional.

In [8]:
# --- LANGKAH 4: LOGIKA OPTIMASI & ATURAN BISNIS ---
def prescribe_strategic_actions(row):
    # Logika Preskriptif Regional untuk AM, AP, AL
    if row['customer_state'] in ['AM', 'AP', 'AL'] and row['late_risk_probability'] > 0.5:
        if row['product_volume_cm3'] > 10000: # Barang besar
            return "📦 STRATEGY: Regional Micro-Hub Placement (Northern Brazil)"
        else:
            return "✈️ STRATEGY: Air-Freight Integration for Small Parcels"
    
    # Logika Biaya Ongkir
    if row['late_risk_probability'] > LogisticsFinanceConfig.CRITICAL_THRESHOLD:
        return "🚀 ACTION: Immediate Premium Carrier Switch"
        
    return "🟢 ACTION: Maintain Standard Operations"

df_prescriptive['prescriptive_recommendation'] = df_prescriptive.apply(prescribe_strategic_actions, axis=1)

5. Impact Dashboarding
Visualisasi dampak preskriptif terhadap performa pengiriman tepat waktu (OTD).

In [9]:
import plotly.express as px

# --- LANGKAH 5: DASHBOARD DAMPAK ---
impact_summary = df_prescriptive.groupby('prescriptive_recommendation').size().reset_index(name='order_count')

fig_impact = px.pie(impact_summary, values='order_count', names='prescriptive_recommendation',
             title="<b>Distribusi Strategi Optimasi Logistik Olist</b>",
             color_discrete_sequence=px.colors.qualitative.Pastel)
fig_impact.update_traces(textposition='inside', textinfo='percent+label')
fig_impact.show()

6. Final Export & Project Handover
Menyimpan hasil akhir untuk digunakan oleh audit_experiment_engine.py dalam otomatisasi operasional.

In [11]:
# --- LANGKAH 6: EKSPOR FINAL & INTEGRASI ---
def finalize_project_02(df, project_root):
    export_path = project_root / "data" / "production" / "06_logistics_prescriptive_recommendations.parquet"
    
    # Menyimpan hanya kolom esensial untuk efisiensi sistem produksi
    final_cols = ['late_risk_probability', 'customer_state', 'prescriptive_recommendation']
    df[final_cols].to_parquet(export_path, index=False)
    
    print("-" * 60)
    print("🏆 PROJECT 02 - LOGISTICS DELIVERY: 100% COMPLETE")
    print(f"📁 Recommendation File: {export_path.name}")
    print("🚀 Status: Ready for Automated Experiment Lab Engine Integration")
    print("-" * 60)

finalize_project_02(df_prescriptive, root)

------------------------------------------------------------
🏆 PROJECT 02 - LOGISTICS DELIVERY: 100% COMPLETE
📁 Recommendation File: 06_logistics_prescriptive_recommendations.parquet
🚀 Status: Ready for Automated Experiment Lab Engine Integration
------------------------------------------------------------


7. Linear Programming (Budget Constraint Optimization)
Sel ini mengidentifikasi kombinasi paket terbaik untuk di-upgrade guna memaksimalkan OTD (On-Time Delivery) dalam batas anggaran tertentu.

In [14]:
import pulp # Library standar industri untuk Linear Programming

# --- LANGKAH 7: LINEAR PROGRAMMING (CONSTRAINT OPTIMIZATION) ---
def execute_constrained_optimization(df, daily_budget=5000):
    """
    Mengoptimalkan pemilihan paket untuk di-upgrade ke Premium Shipping
    Target: Memaksimalkan 'Late Cost Saved' dengan batasan 'Daily Budget'.
    """
    # 1. Menyiapkan parameter simulasi
    df_opt = df[df['late_risk_probability'] > 0.5].copy()
    
    # Cost to upgrade vs Potential Loss (Churn + Penalty)
    df_opt['upgrade_cost'] = 15.0 # Biaya tetap premium shipping
    df_opt['potential_loss'] = 60.0 # Churn Cost (50) + Penalty (10)
    
    # Menghitung Expected Saving: Probabilitas Terlambat * Potential Loss
    df_opt['expected_saving'] = df_opt['late_risk_probability'] * df_opt['potential_loss']
    
    # 2. Inisialisasi Masalah Optimasi (Maximize Profit/Saving)
    prob = pulp.LpProblem("Logistics_Budget_Optimization", pulp.LpMaximize)
    
    # Variabel Keputusan: x[i] = 1 jika di-upgrade, 0 jika tidak
    order_ids = df_opt.index.tolist()
    x = pulp.LpVariable.dicts("upgrade", order_ids, cat=pulp.LpBinary)
    
    # Objective Function: Memaksimalkan total expected saving
    prob += pulp.lpSum([x[i] * df_opt.loc[i, 'expected_saving'] for i in order_ids])
    
    # Constraint: Total biaya upgrade tidak boleh melebihi daily_budget
    prob += pulp.lpSum([x[i] * df_opt.loc[i, 'upgrade_cost'] for i in order_ids]) <= daily_budget
    
    # 3. Solve Masalah
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    
    # 4. Integrasi Hasil
    df_opt['lp_decision'] = [pulp.value(x[i]) for i in order_ids]
    
    # Summary untuk Stakeholders
    orders_to_upgrade = df_opt['lp_decision'].sum()
    total_savings = sum(pulp.value(x[i]) * df_opt.loc[i, 'expected_saving'] for i in order_ids)
    
    print("-" * 60)
    print(f"🎯 OPTIMASI BERHASIL (Budget Limit: R$ {daily_budget:,})")
    print(f"✅ Rekomendasi Upgrade : {int(orders_to_upgrade)} Paket")
    print(f"✅ Total Potensi Saving : R$ {total_savings:,.2f}")
    print(f"✅ Efisiensi Budget    : {(total_savings / daily_budget):.2f}x ROI")
    print("-" * 60)
    
    return df_opt

df_constrained = execute_constrained_optimization(df_prescriptive)

------------------------------------------------------------
🎯 OPTIMASI BERHASIL (Budget Limit: R$ 5,000)
✅ Rekomendasi Upgrade : 333 Paket
✅ Total Potensi Saving : R$ 14,502.70
✅ Efisiensi Budget    : 2.90x ROI
------------------------------------------------------------


# 04. Rekomendasi Preskriptif & Optimasi Alokasi Anggaran

Tahap Preskriptif ini merupakan puncak dari proyek logistik, di mana kita mensimulasikan tindakan strategis untuk meminimalkan kerugian finansial akibat keterlambatan pengiriman pada ekosistem Olist.

### 🎯 Hasil Optimasi & Solusi Strategis
1. **Peningkatan Efisiensi Biaya**: Melalui teknik *Linear Programming*, kita berhasil mengidentifikasi alokasi anggaran optimal untuk *Premium Shipping* yang memberikan ROI tertinggi, menyelamatkan potensi kerugian *churn* pelanggan.
2. **Intervensi Risiko Kritis**: Sistem secara otomatis merekomendasikan penggantian kurir (*Carrier Switch*) pada pesanan dengan probabilitas keterlambatan > 80%, terutama untuk kategori produk bervolume besar.
3. **Optimasi Regional**: Hasil analisis merekomendasikan penempatan stok strategis (*Regional Micro-Hub*) di wilayah Utara (AM, AP, AL) untuk memangkas *lead time* secara permanen tanpa terus-menerus bergantung pada pengiriman ekspres yang mahal.

### 📊 Dampak Operasional
* **Target OTD (On-Time Delivery)**: Proyeksi peningkatan ketepatan waktu pengiriman secara signifikan pada wilayah *bottleneck*.
* **Skalabilitas Keputusan**: Logika preskriptif yang dibangun bersifat modular dan siap diintegrasikan ke dalam sistem manajemen transportasi (TMS) Olist.

---

# 🚀 Next Stage: 05_automation_experiment_lab.ipynb

Setelah menyelesaikan seluruh siklus analitik (Deskriptif, Diagnostik, Prediktif, dan Preskriptif), tahap selanjutnya adalah melakukan pengujian skala kecil untuk memvalidasi efektivitas strategi kita.

**Fokus Automation Experiment Lab:**
* **A/B Testing Simulation**: Membandingkan performa grup kontrol (tanpa optimasi) vs grup eksperimen (dengan optimasi preskriptif).
* **Stress Test Pipeline**: Menguji ketahanan *automation engine* dalam menangani lonjakan data transaksi secara *real-time*.
* **Refinement Logic**: Menyesuaikan parameter biaya dan *threshold* risiko berdasarkan umpan balik hasil eksperimen sebelum implementasi produksi penuh.

---
**Status Audit**: Prescriptive Analysis Selesai | **Level**: Platinum Standard | **Timestamp**: 2026-02-08